In [ ]:
#!pip install biopython pandas matplotlib seaborn pyarrow openpyxl umap-learn plotly

In [ ]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import scipy as sp
import pandas as pd
import networkx as nx
import random
import matplotlib.pyplot as plt
# import warnings filter
from warnings import simplefilter
# ignore all future warnings
simplefilter(action='ignore', category=FutureWarning)
from IPython.core.display import HTML
HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell;text-align: center;vertical-align: middle;}</style>')

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="umap")

In [ ]:

from amr.debiasing import (
    build_pathogen_vectors_antibiotic,
    build_pathogen_vectors_class,
    debias_amr_marginal,
)
from amr.visualization import (
    _umap_scatter,
    plot_heatmap_counts_vs_expected_antibiotic,
    plot_heatmap_counts_vs_expected_class,
    plot_heatmap_for_pathogen,
    plot_heatmap_for_pathogen_debiased,
    plot_umap_antibiotic_activity,
)


---

In [ ]:
data_df = pd.read_parquet("data.parquet")
data_df

In [ ]:
antibiotics_df = pd.read_excel("antibiotic_list2.xlsx")
for col in antibiotics_df.columns:
    col_list = antibiotics_df[col].values
    print(f'#{col}: {len(set(col_list))}')
antibiotics_df.head()

In [ ]:
amr_df = pd.read_parquet("pmids_results_positive.parquet")
amr_df

In [ ]:
pmid_pattern = r"(\d{4,})"  # capture group

unique_pmids = (
    amr_df["pmids"]
    .dropna()
    .astype(str)                 # ensure string dtype
    .str.extractall(pmid_pattern)[0]
    .nunique()
)

print(f"Unique PMIDs: {unique_pmids:,}")


In [ ]:
antibiotic = 'Amikacin'
date = ['1975/01/01','1980/01/01','1985/01/01','1990/01/01','1995/01/01','2000/01/01','2005/01/01','2010/01/01','2015/01/01','2020/01/01']
embedding_column = "anonymised_embedding"

In [ ]:
_ = plot_umap_antibiotic_activity(
    data_df,
    amr_df,
    antibiotic=antibiotic,
    date='2020/01/01',
    embedding_column=embedding_column,
    cmap="Reds",
    min_n_causation=3,
    max_marker_size=5,
    min_marker_size=1,
    image_size=(1700,400)
)

In [ ]:
_ = plot_umap_antibiotic_activity(
    data_df,
    amr_df,
    antibiotic=antibiotic,
    date=date,
    embedding_column=embedding_column,
    cmap="Reds",
    min_n_causation=3,
    max_marker_size=5,
    min_marker_size=1,
    image_size=(700,400)
)

In [ ]:
mp4_path = f"timelapse_{antibiotic}.mp4"
image_size= ( 704*2, 512*2)
# Generate and display an MP4 animation for a sequence of dates
try:
    _ = plot_umap_antibiotic_activity(
        data_df,
        amr_df,
        antibiotic=antibiotic,
        date=date,
        embedding_column=embedding_column,
        cmap="Reds",
        min_n_causation=3,
        max_marker_size=7,
        min_marker_size=0.5,
        mp4_path=mp4_path,
        video_total_duration=5.0,
        video_ping_pong=False,   # set False for continuous loop
        image_size=image_size,
    )
    from IPython.display import HTML, display
    import os
    if os.path.exists(mp4_path):
        display(HTML(f'<video width="{image_size[0]}" controls loop><source src="{mp4_path}" type="video/mp4"></video>'))
    else:
        print("Animation not found; run the generation cell first.")
    print(f"Saved animation to {mp4_path}")
except ImportError as e:
    print("Install imageio, kaleido, and imageio-ffmpeg to enable animation export:", e)


### Debiasing via marginal interval profiles

We estimate expected resistance signal for each pathogen–antibiotic pair by isolating how each antibiotic behaves across intervals.

1. Assign every interval to a interval and compute log-damped counts
   $$L_{p,a,t} = \log(1 + C_{p,a,t}).$$
2. Aggregate the log counts across pathogens for each antibiotic (or class) to obtain its interval profile. Normalise this profile so it sums to one — this captures the typical interval distribution for that antibiotic.
3. Sum the interval-level log counts for each pathogen–antibiotic pair to obtain the total log signal
   $$L^{\text{tot}}_{p,a} = \sum_t L_{p,a,t},$$
   which reflects how much attention that pair receives overall after damping spikes.
4. Allocate the total log signal back to intervals using the antibiotic's marginal profile to obtain the expected log signal
   $$\hat{L}_{p,a,t} = L^{\text{tot}}_{p,a} \times P_{a,t}.$$
5. Convert the expected log signal to the count scale with
   $$\exp(\hat{L}_{p,a,t}) - 1$$
   to obtain the expected publication count per interval.
6. Analyse the log-space residuals
   $$R_{p,a,t} = L_{p,a,t} - \hat{L}_{p,a,t}$$
   to highlight intervals where the pair appears more or less often than the antibiotic's historical pattern suggests.

In the heatmaps we annotate each cell with the observed and expected counts, and colour by the residual in log space.




In [ ]:
marg_ab, marg_cl = debias_amr_marginal(amr_df, antibiotics_df)


### Pathogen vector representations

The helpers below compress each pathogen into a normalised log-residual vector
at the antibiotic and class levels, alongside total log-count weights.


In [ ]:
pathogen_names_ab, pathogen_log_sums_ab, pathogen_vectors_ab = build_pathogen_vectors_antibiotic(
    amr_df,
    antibiotics_df,
)

pathogen_names_cl, pathogen_log_sums_cl, pathogen_vectors_cl = build_pathogen_vectors_class(
    amr_df,
    antibiotics_df,
)

pathogen_vectors_ab.shape, pathogen_vectors_cl.shape



### UMAP scatter plots

Two-dimensional UMAP projections for antibiotic and class vectors, coloured by pathogen genus.


In [ ]:
fig_ab = _umap_scatter(
    pathogen_names_ab,
    pathogen_log_sums_ab,
    pathogen_vectors_ab,
    title="UMAP projection of pathogens (antibiotic space)",
    marker_max_size=10,
    figure_size=(800, 600),
    color_map="Rainbow",
    min_size=50.0,
)
fig_ab



In [ ]:
fig_cl = _umap_scatter(
    pathogen_names_cl,
    pathogen_log_sums_cl,
    pathogen_vectors_cl,
    title="UMAP projection of pathogens (class space)",
    min_size=0.0,
)
fig_cl



### Next steps: forecasting counts from embeddings

**Goal:** produce pathogen–antibiotic (and class-level) count forecasts for a future decade by combining forecasts of global antibiotic behaviour with pathogen-specific residual structure, while denormalising using observed totals from training.

1. **Forecast antibiotic/class marginal profiles.**  Fit a simple linear trend (or polynomial with regularisation) on each antibiotic’s decade-wise marginal signal $P_{a,t}$ to extrapolate $P_{a,t+1}$.  We treat these projections as scaled intensity factors rather than probabilities, so no renormalisation step is required even if total mass shifts.
2. **Assemble training targets in normalised form.**  Continue using the residual embeddings (normalised log differences) derived from observed data so the model learns deviations relative to the historical marginal behaviour.
3. **Model pathogen residual vectors.**  Train a gradient-boosted model (e.g. LightGBM/XGBoost) that consumes a window of the past $k$ decades (concatenated embeddings, optional covariates) and predicts the residual pattern for the next decade $R_{p,a,t+1}$.  Because the data are detrended, learning joint temporal patterns across pathogens should generalise across periods.
4. **Reconstruct predicted log signals.**  Combine the extrapolated marginal scale and the predicted residuals to produce normalised log values for the future decade.  Instead of forecasting new totals, re-introduce scale by multiplying with the observed totals from the latest available decade (or another chosen reference).
5. **Recover counts.**  Apply $\exp(\hat{L}_{p,a,t+1}) - 1$ to obtain count estimates, clip negatives, and optionally round.  Because we anchored to historical totals, the resulting sums reflect past magnitude while accommodating new marginal trends.

**Implementation plan:**
- Build decade-wise panels for antibiotic/class marginals, pathogen totals, and residual embeddings (training features/targets).
- Prototype the linear-trend extrapolators (`numpy.polyfit`/`LinearRegression`) for the marginal signals; inspect whether extrapolated values remain in plausible ranges without renormalisation.
- Train and validate gradient-boosted models that take $k$-decade windows of residual embeddings (plus metadata) and predict the next-decade residuals; tune $k$ and feature engineering for best generalisation.
- Integrate the reconstruction workflow that reuses observed totals for de-normalisation; back-test on held-out decades to measure reconstruction error relative to ground truth counts.
- Package the pipeline into reusable helpers/notebook cells to enable rapid experimentation with alternative temporal models or denormalisation baselines.



In [ ]:
pathogen = 'Neisseria gonorrhoeae'

In [ ]:
pathogen = 'Mycobacterium senegalense'

In [ ]:
pathogen = 'Staphylococcus aureus'

In [ ]:
class_columns = ['Class', 'Simplified categories', 'High level categories', 'Origin']
for class_column in class_columns:
    plot_heatmap_counts_vs_expected_class(
        marg_cl,
        pathogen,
        class_column=class_column,
        class_df=antibiotics_df,
        figsize=(15, 10)
    )


In [ ]:
# Antibiotic-level heatmap
plot_heatmap_counts_vs_expected_antibiotic(
    marg_ab,
    pathogen,
    class_df=antibiotics_df,
    figsize=(14, 50)
)



In [ ]:
print(f"Sampled pathogen (log-weighted): {pathogen}")

# Plot heatmaps
df = amr_df[amr_df['pathogen'] == pathogen]
df.groupby('end_date')['count'].sum().plot(figsize=(14, 2))


In [ ]:
class_columns = ['Class', 'Simplified categories', 'High level categories', 'Origin']
for class_column in class_columns:
    plot_heatmap_for_pathogen(
        amr_df, 
        pathogen,
        class_df=antibiotics_df, 
        group_column=class_column,
        show_only_class=True, 
        sort_by_frequency=False, 
        figsize=(10,9))


In [ ]:
class_columns = ['Class', 'Simplified categories', 'High level categories', 'Origin']
plot_heatmap_for_pathogen(amr_df, pathogen, 
                          class_df=antibiotics_df, group_column='High level categories', 
                          show_only_class=False, figsize=(13,50))

---